# 0. Configuration

In [276]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [277]:
import pandas as pd 
import numpy as np

In [278]:
from scipy.optimize import minimize, OptimizeResult
from statsmodels.discrete.discrete_model import Logit

In [279]:
from core.dgp import *
from core.estimators import BaseEstimator

# 1. Group Targeting

In [280]:
class DGP1:
    """ 
    Data Generating Process 1:
    \tau_i \in \{a_1, \dots, a_K\}  # K groups, each with a different treatment effect a_k 
    Y_i = \tau_i T_i + \epsilon_i, \epsilon_i \sim N(0, \sigma^2)  # outcome model
    T_i \sim Bernoulli(p)  # treatment assignment model
    """
    def __init__(
        self, num_groups: int, te_diff: float, group_func: callable, 
        noise_std: float = 1.0, treatment_assign_prob: float = 0.5, **kwargs
    ):
        """  
        DGP for model 1

        Params:
        -------
        num_groups: int, number of groups
        te_diff: float, treatment effect difference
        noise_std: float, standard deviation of the noise in the outcome model
        treatment_assign_prob: float, probability of treatment assignment
        """
        # attributes
        self.n_groups = num_groups
        self.te_diff = te_diff
        self.te_list = [1 + i * te_diff for i in range(num_groups)]
        self.group_func = group_func
        self.noise_std = noise_std
        self.treatment_assign_prob = treatment_assign_prob

    def generate_training_data(self, sample_size: int, seed=None) -> tuple:
        """
        Generate training data

        Params:
        -------
        sample_size: int, number of samples to generate

        Returns:
        -------
        tuple: 
            - X: np.ndarray, covariate
            - T: np.ndarray, treatment assignment
            - Y: np.ndarray, outcome
        """
        # set seed
        seed = seed if seed else np.random.randint(0, 1e6)
        np.random.seed(seed)

        # treatment effect function: given a covariate x, return the treatment effect of the group
        te_func = lambda x: self.te_list[self.group_func(x)]
        
        # generate data
        X = self.__generate_individual_characteristics(sample_size, seed)  # (sample_size, )
        T = np.random.binomial(1, 0.5, sample_size)  # (sample_size, )
        te_arr = np.array([te_func(x) for x in X])  # (sample_size, )
        Y = te_arr * T  + np.random.normal(0, self.noise_std, sample_size)  # (sample_size, )
        
        return X.reshape(-1, 1), T.reshape(-1, 1), Y.reshape(-1, 1)  # (sample_size, 1)
    
    def generate_testing_data(self, sample_size: int, seed=None) -> np.ndarray:
        seed = seed if seed else np.random.randint(0, 1e6)
        
        # generate data
        X = self.__generate_individual_characteristics(sample_size, seed) # (sample_size, )

        return X  # (sample_size, )

    def __generate_individual_characteristics(self, sample_size: int, seed: int) -> np.ndarray:
        np.random.seed(seed)
        return np.random.uniform(-3, 3, sample_size)

# 2. Personalized Pricing

Based on (Dube and Misra 2023, Journal of Political Economy)

In [281]:
class PersonalizedPricingDGP(object):
    def __init__(self, cov_dim: int = 1):
        self.cov_dim = cov_dim
        self.util_const_map = np.random.uniform(0, 1, size=(cov_dim, 1))  # (cov_dim, 1)
        self.util_price_map = np.random.uniform(-0.1, 0, size=(cov_dim, 1))  # (cov_dim, 1)

    def generate_training_data(
        self, sample_size: int, price_lb: float, price_ub: float, price_diff: float, seed=None
    ) -> tuple:
        """
        Generate training data

        Params:
        -------
        sample_size: int, number of samples to generate
        price_lb: float, lower bound of the price
        price_ub: float, upper bound of the price
        price_diff: float, price difference
        seed: int, random seed

        Returns:
        -------
        tuple: 
            - X: np.ndarray, covariate
            - T: np.ndarray, treatment assignment
            - Y: np.ndarray, outcome
        """
        # generate data
        # individual characteristics
        X_arr = self.__generate_individual_characteristics(sample_size)  # (sample_size, cov_dim)
        
        # price (treaments)
        price_arr = np.random.choice(np.arange(price_lb, price_ub, price_diff), sample_size).reshape(-1, 1)  # (sample_size, 1)

        # error
        err_arr = np.random.gumbel(loc=0, scale=1, size=(sample_size, 1))  # (sample_size, 1)

        # utility: (sample_size, 1)
        util_arr = X_arr @ self.util_const_map + X_arr @ self.util_price_map * price_arr + err_arr

        # demand: (sample_size, 1), buy if utility > 0, else don't buy.
        demand_arr = (util_arr > 0).astype(int)

        return X_arr, price_arr, demand_arr
    
    def generate_testing_data(self, sample_size: int) -> np.ndarray:        
        return self.__generate_individual_characteristics(sample_size)  # (sample_size, )

    def __generate_individual_characteristics(self, sample_size: int) -> np.ndarray:
        return np.random.normal(loc=3, scale=0.5, size=(sample_size, self.cov_dim))

In [282]:
dgp = PersonalizedPricingDGP(cov_dim=1)

In [295]:
cov_arr, price_arr, demand_arr = dgp.generate_training_data(
    sample_size=1000, price_lb=1, price_ub=10, price_diff=1
)

In [304]:
dgp.util_const_map[0, 0], dgp.util_price_map[0, 0]

(0.006928049725869911, -0.06228884855677578)

In [308]:
exog = np.concatenate([cov_arr, price_arr * cov_arr], axis=1)

model = Logit(demand_arr, exog).fit()

model.summary()


Optimization terminated successfully.
         Current function value: 0.601115
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      998
Method:                           MLE   Df Model:                            1
Date:                Wed, 05 Jun 2024   Pseudo R-squ.:                 0.07244
Time:                        09:58:02   Log-Likelihood:                -601.12
converged:                       True   LL-Null:                       -648.06
Covariance Type:            nonrobust   LLR p-value:                 3.325e-22
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.2006      0.050      4.028      0.000       0.103       0.298
x2            -0.0857      0.010     -8.850      0.000      -0.105      -0.067
==============================================================================
"""

In [309]:
from sklearn.linear_model import LogisticRegression

In [310]:
model = LogisticRegression()
model.fit(exog, demand_arr.flatten())

LogisticRegression()

In [316]:
model.coef_.T.shape

(2, 1)

In [312]:
type(model)

sklearn.linear_model._logistic.LogisticRegression